Perfect! Let’s break down the **advanced RAG search techniques** shown in your screenshot, explain each one with examples, and add a few **additional advanced techniques** that are widely used in production.

---

# 1️⃣ **Hybrid Search**

**Definition:** Combines **semantic search** (vector similarity) with **keyword search** (like BM25) to improve coverage.

**Why:**

* Semantic search understands meaning → “return policy” ≈ “refund rules”
* Keyword search ensures exact matches are not missed → “sneakers return policy”

**Example:**

```python
from langchain.vectorstores import Chroma
from rank_bm25 import BM25Okapi

# 1. Retrieve top results using vector similarity
vector_results = vectorstore.similarity_search("sneakers return policy", k=10)

# 2. Retrieve top results using BM25 keywords
bm25 = BM25Okapi(corpus)
bm25_results = bm25.get_top_n("sneakers return policy", corpus, n=10)

# 3. Combine results (hybrid)
final_results = merge(vector_results, bm25_results)
```

---

# 2️⃣ **Re-ranking**

**Definition:**

* First, retrieve **many documents quickly**
* Then, **re-rank top documents using a specialized model** for higher precision

**Steps:**

1. First pass → Retrieve top 100 (fast, approximate)
2. Second pass → Re-rank top 10 using a **cross-encoder model** (precise)

**Example:**

```python
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = cross_encoder.predict([(query, doc) for doc in top_100_docs])
top_docs = select_top_n(scores, top_10=10)
```

---

# 3️⃣ **Metadata Filtering**

**Definition:** Filter documents using **metadata fields** before or during search.

**Why:** Improves relevance and reduces noise.

**Example:**

* Filter by `date`, `author`, `category`
* Only search documents in **“FAQ 2025”** or authored by **“Legal Team”**

```python
vector_results = vectorstore.similarity_search(
    query="return policy", 
    k=10, 
    filter={"category": "FAQ", "year": 2025}
)
```

---

# 4️⃣ **Maximal Marginal Relevance (MMR)**

**Definition:** Balances **relevance** with **diversity** to avoid redundant results.

**Why:**

* Normal similarity search often returns similar/duplicate results
* MMR ensures you get **different angles** of information

**Example:**

```python
from langchain.vectorstores import Chroma

docs = vectorstore.max_marginal_relevance_search(
    "return policy for sneakers", 
    k=10, 
    fetch_k=50
)
```

* `fetch_k` → fetch more docs for MMR to select diverse top 10

---

# 5️⃣ **Additional Advanced Techniques**

### 5.1 **Contextual Query Expansion**

* Automatically expand query with synonyms, LLM suggestions, or embeddings
* Example: "return policy" → "return policy, refund, exchange"

### 5.2 **Query-Document Feedback Loops**

* Re-rank based on previous user interactions
* Improves results over time with relevance feedback

### 5.3 **Chunking + Overlap**

* Break large documents into overlapping chunks
* Ensures RAG retrieves all relevant context without losing info

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(large_document)
```

### 5.4 **Vector Store Ensembles**

* Combine multiple vector stores (FAISS, Chroma, Pinecone) for hybrid performance
* Useful if you want **fast local search + persistent cloud search**

### 5.5 **Temporal / Context-Aware Filtering**

* Filter based on **time** or **versioning**
* E.g., only use policies valid **after 2023**

---

# ✅ Summary Table

| Technique                    | Purpose                     | Example / Use               |
| ---------------------------- | --------------------------- | --------------------------- |
| Hybrid Search                | Combine semantic + keyword  | Vector search + BM25        |
| Re-ranking                   | Improve precision           | Cross-encoder top-10        |
| Metadata Filtering           | Filter by attributes        | category, author, date      |
| Maximal Marginal Relevance   | Avoid redundancy            | MMR selection               |
| Query Expansion              | Broader search coverage     | Synonyms or LLM suggestions |
| Feedback Loops               | Improve relevance over time | User click data             |
| Chunking + Overlap           | Ensure complete context     | Split large docs            |
| Vector Store Ensembles       | Best of multiple DBs        | Local + cloud hybrid        |
| Temporal / Context Filtering | Use current context         | Only latest docs            |

---

If you want, I can **draw a full production-ready RAG pipeline diagram** with **all these techniques integrated**, showing how **Pinecone / FAISS / Chroma + LLMs + Re-ranking + MMR + Metadata** work together — ready for your Adidas or policy RAG project.

Do you want me to do that next?


Absolutely! Let’s start from **scratch** and explain everything clearly, including the **BM25 corpus** concept, the full advanced RAG techniques, examples, and additional production techniques. I’ll also structure it so it’s **ready to implement**.

---

# 1️⃣ **BM25 and Corpus**

**BM25** is a **keyword-based retrieval algorithm** (classic IR technique) used to score documents based on the query. It works **without embeddings** — purely based on term frequency, inverse document frequency, and document length.

* **Corpus:** The collection of documents you want to search.
* **BM25 Object:** `BM25Okapi(corpus)` initializes the BM25 model for your corpus.

### Example:

```python
from rank_bm25 import BM25Okapi

# 1. Your document corpus
corpus = [
    "The return policy for sneakers is 30 days.",
    "Refund rules for apparel may vary.",
    "You can exchange shoes within 15 days of purchase.",
]

# 2. Initialize BM25 with corpus
bm25 = BM25Okapi(corpus)

# 3. Query the corpus
query = "sneaker return policy"
top_docs = bm25.get_top_n(query, corpus, n=2)

print(top_docs)
```

**Output:**

```
["The return policy for sneakers is 30 days.", "You can exchange shoes within 15 days of purchase."]
```

> BM25 gives you **keyword-relevant results** fast.

---

# 2️⃣ **Hybrid Search**

**Definition:** Combines **semantic vector search** + **BM25 keyword search**.

**Why:** Semantic search captures meaning; BM25 ensures exact keyword matches are included.

```python
from langchain.vectorstores import Chroma

# Vector search (semantic)
vector_results = vectorstore.similarity_search("sneakers return policy", k=10)

# BM25 search (keyword)
bm25_results = bm25.get_top_n("sneakers return policy", corpus, n=10)

# Combine results (hybrid)
final_results = vector_results + bm25_results
```

---

# 3️⃣ **Re-ranking**

**Definition:**

* First retrieve **many documents quickly** (fast, approximate)
* Then re-rank **top documents** for precision using a **cross-encoder or LLM**

```python
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = cross_encoder.predict([(query, doc) for doc in top_100_docs])
top_docs = select_top_n(scores, top_10=10)
```

> Two-pass retrieval ensures **speed + accuracy**.

---

# 4️⃣ **Metadata Filtering**

**Definition:** Filter documents by metadata **before or during search**.

```python
vector_results = vectorstore.similarity_search(
    query="return policy", 
    k=10, 
    filter={"category": "FAQ", "year": 2025, "author": "Legal Team"}
)
```

> Only documents that match your filter are searched → reduces noise.

---

# 5️⃣ **Maximal Marginal Relevance (MMR)**

**Definition:** Balances **relevance** + **diversity** to avoid duplicate/redundant results.

```python
docs = vectorstore.max_marginal_relevance_search(
    "return policy for sneakers", 
    k=10,  # final top 10
    fetch_k=50  # fetch 50 candidates for MMR selection
)
```

> Ensures you get **different angles of relevant information**.

---

# 6️⃣ **Additional Advanced Techniques**

| Technique                    | Purpose                                       | Example                                                            |
| ---------------------------- | --------------------------------------------- | ------------------------------------------------------------------ |
| Query Expansion              | Expand query with synonyms or LLM suggestions | "return policy" → "return policy, refund, exchange"                |
| Feedback Loops               | Re-rank based on user interactions            | Use click data to boost ranking                                    |
| Chunking + Overlap           | Split large docs to maintain context          | `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` |
| Vector Store Ensembles       | Combine multiple vector DBs                   | Local FAISS + Pinecone cloud                                       |
| Temporal / Context Filtering | Use only valid/current docs                   | Filter by date or version                                          |

---

# 7️⃣ **Full Hybrid RAG Flow Example**

```python
from rank_bm25 import BM25Okapi
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from sentence_transformers import CrossEncoder

# Corpus
corpus = [
    "The return policy for sneakers is 30 days.",
    "Refund rules for apparel may vary.",
    "You can exchange shoes within 15 days of purchase.",
]

# BM25
bm25 = BM25Okapi(corpus)

# Vector store setup
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(persist_directory="db", embedding_function=embeddings)
vectorstore.add_texts(corpus)

query = "sneakers return policy"

# Hybrid retrieval
vector_results = vectorstore.similarity_search(query, k=10)
bm25_results = bm25.get_top_n(query, corpus, n=10)
hybrid_results = vector_results + bm25_results

# Re-ranking top 10
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = cross_encoder.predict([(query, doc) for doc in hybrid_results])
top_docs = [doc for _, doc in sorted(zip(scores, hybrid_results), reverse=True)[:10]]

# Metadata filtering example
filtered_docs = [doc for doc in top_docs if "FAQ" in doc]  # simple metadata filter

# MMR for diversity
final_docs = vectorstore.max_marginal_relevance_search(
    query, k=5, fetch_k=20
)

print(final_docs)
```

> This shows a **full end-to-end advanced retrieval pipeline**:
> BM25 + vector search → hybrid → re-ranking → metadata filter → MMR → final top docs.

---

If you want, I can now **draw a clear diagram showing the “Hybrid RAG + BM25 + Re-ranking + MMR + Metadata filtering + Chunking + Feedback loop” pipeline** — ready for **production use**.

Do you want me to do that next?
